# py-sigxtalk Benchmark: Python vs R Comparison

This notebook benchmarks the Python `pysigxtalk` pipeline against the R `SigXTalkR` package.
It measures timing and compares PRS correlation between Python and R implementations.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
import sys
sys.path.insert(0, '../src')
import pysigxtalk as psx
import anndata
print(f"py-sigxtalk version: {psx.__version__}")

## Step 1: Load PBMC3k Data & Databases

In [ ]:
adata = anndata.read_h5ad('pbmc3k_final.h5ad')
print(f"PBMC3k: {adata.shape[0]} cells, {adata.shape[1]} genes")
print(f"Cell types: {adata.obs['cell_type'].value_counts().to_dict()}")

rtf_db, tftg_db = psx.load_databases(species="human")
print(f"RTF: {len(rtf_db)}, TFTG: {len(tftg_db)}")

## Step 2: Prepare Data for CD14+ Mono

In [ ]:
target_type = 'CD14+ Mono'
target_mask = adata.obs['cell_type'] == target_type
adata_target = adata[target_mask].copy()

# Expression matrix (genes x cells)
if 'scale_data' in adata_target.layers:
    X = adata_target.layers['scale_data']
else:
    X = adata_target.X
exp_mat = pd.DataFrame(
    X.T.toarray() if hasattr(X, 'toarray') else X.T,
    index=adata_target.var.index.tolist(),
    columns=adata_target.obs.index.tolist()
)
exp_mat = psx.get_exp_clu(exp_mat, cutoff=0.1)
print(f"Expression: {exp_mat.shape}")

# Target genes
if 'highly_variable' in adata.var.columns:
    target_genes = [g for g in adata.var[adata.var['highly_variable']].index if g in exp_mat.index]
else:
    target_genes = exp_mat.var(axis=1).nlargest(200).index.tolist()
print(f"Target genes: {len(target_genes)}")

# Synthetic LR pairs
all_genes = exp_mat.index.tolist()
receptors_in_data = [g for g in rtf_db['from'].unique() if g in all_genes][:20]
np.random.seed(42)
lr_pairs = pd.DataFrame({
    'Ligand': np.random.choice(receptors_in_data, 30, replace=True),
    'Receptor': np.random.choice(receptors_in_data, 30, replace=True),
    'Weight': np.random.rand(30) * 0.8 + 0.2,
})
print(f"LR pairs: {len(lr_pairs)}")

## Step 3: Benchmark Python Pipeline

In [ ]:
# Preprocessing
t0 = time.time()
inputs = psx.prepare_input(
    exp_mat=exp_mat, target_genes=target_genes,
    lr_pairs=lr_pairs, rtf_db=rtf_db, tftg_db=tftg_db,
)
t_preprocess = time.time() - t0
print(f"Preprocessing: {t_preprocess:.2f}s")
print(f"  RTF: {len(inputs.rtf_filtered)}, TFTG: {len(inputs.tftg_filtered)}")

In [ ]:
# HGNN
t0 = time.time()
pathways = psx.run_hgnn(
    inputs, epochs=30, device="cpu", seed=42,
    hgnn_dims=[64, 32], linear_dims=[16, 8]
)
t_hgnn = time.time() - t0
print(f"HGNN: {t_hgnn:.2f}s")
print(f"  Pathways: {len(pathways)}")
print(f"  Active (>0.5): {len(pathways[pathways['pred_label'] > 0.5])}")

In [ ]:
# PRS (sklearn)
t0 = time.time()
prs_sklearn = psx.compute_prs(inputs.exp_clu, pathways, engine="sklearn", n_estimators=100, cutoff=0.5)
prs_filtered = psx.filter_results(prs_sklearn, prs_threshold=0.01)
t_prs_sklearn = time.time() - t0
print(f"PRS (sklearn): {t_prs_sklearn:.2f}s, {len(prs_filtered)} pathways")

In [ ]:
# PRS timing only (LightGBM comparison removed)
print(f"Python PRS: {t_prs_sklearn:.2f}s")

## Step 4: Timing Comparison

In [ ]:
## Step 4: Timing Comparison

```python
fig, ax = plt.subplots(figsize=(8, 5))
steps = ["PRS (Python)"]
times = [t_prs_sklearn]

if t_prs_r > 0:
    steps.append("PRS (R)")
    times.append(t_prs_r)

colors = ["steelblue", "coral"]
ax.barh(steps, times, color=colors)
ax.set_xlabel("Time (seconds)")
ax.set_title("Python vs R PRS Timing (PBMC3k CD14+ Mono)")
for i, v in enumerate(times):
    ax.text(v + 0.1, i, f"{v:.2f}s", va="center")
plt.tight_layout()
plt.savefig("benchmark_timing.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Python: {t_prs_sklearn:.2f}s")
if t_prs_r > 0:
    print(f"R: {t_prs_r:.2f}s")
    print(f"Speedup: {t_prs_r/t_prs_sklearn:.2f}x")
```

## Step 5: Run R PRS via rpy2

```python
# Try to run R PRS using rpy2
try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.packages import importr
    pandas2ri.activate()
    
    sigxtalk_r = importr('SigXTalkR')
    
    # Export expression matrix for R
    exp_clu_r = pandas2ri.py2rpy(inputs.exp_clu)
    pathways_r = pandas2ri.py2rpy(pathways[pathways['pred_label'] > 0.5])
    
    # Run R PRS
    t0 = time.time()
    prs_r = sigxtalk_r.PRS_calc(exp_clu_r, pathways_r, cutoff=0.1)
    t_prs_r = time.time() - t0
    
    # Convert back to pandas
    prs_r_df = pandas2ri.rpy2py(prs_r)
    prs_r_filtered = psx.filter_results(prs_r_df, prs_threshold=0.01)
    
    print(f"PRS (R): {t_prs_r:.2f}s, {len(prs_r_filtered)} pathways")
except Exception as e:
    print(f"R PRS failed: {e}")
    t_prs_r = 0
    prs_r_filtered = pd.DataFrame()
```

In [ ]:
## Step 6: Correlation Analysis (Python vs R)

```python
if len(prs_filtered) > 0 and len(prs_r_filtered) > 0:
    # Merge on Receptor-SSC-Target
    merged = prs_filtered.merge(prs_r_filtered, on=["Receptor", "SSC", "Target"], suffixes=("_py", "_r"))
    
    if len(merged) > 0:
        corr_pearson, pval_p = pearsonr(merged["Weight_py"], merged["Weight_r"])
        corr_spearman, pval_s = spearmanr(merged["Weight_py"], merged["Weight_r"])
        
        print(f"Matching pathways: {len(merged)}")
        print(f"Pearson correlation: {corr_pearson:.4f} (p={pval_p:.2e})")
        print(f"Spearman correlation: {corr_spearman:.4f} (p={pval_s:.2e})")
        
        # Visualization 1: Scatter plot
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Scatter
        ax = axes[0]
        ax.scatter(merged["Weight_py"], merged["Weight_r"], alpha=0.5, s=10, c="steelblue")
        ax.plot([0, merged["Weight_py"].max()], [0, merged["Weight_py"].max()], 'r--', label='y=x')
        ax.set_xlabel("Python PRS")
        ax.set_ylabel("R PRS")
        ax.set_title(f"Python vs R PRS (r={corr_pearson:.4f})")
        ax.legend()
        
        # Distribution comparison
        ax = axes[1]
        ax.hist(prs_filtered["Weight"], bins=50, alpha=0.5, label="Python", color="steelblue")
        ax.hist(prs_r_filtered["Weight"], bins=50, alpha=0.5, label="R", color="coral")
        ax.set_xlabel("PRS Weight")
        ax.set_ylabel("Frequency")
        ax.set_title("PRS Distribution Comparison")
        ax.legend()
        
        plt.tight_layout()
        plt.savefig("benchmark_correlation.png", dpi=150, bbox_inches="tight")
        plt.show()
        
        # Visualization 2: Difference distribution
        fig, ax = plt.subplots(figsize=(8, 5))
        diff = merged["Weight_py"] - merged["Weight_r"]
        ax.hist(diff, bins=50, color="steelblue", edgecolor="black")
        ax.axvline(x=0, color='red', linestyle='--')
        ax.set_xlabel("Python - R Difference")
        ax.set_ylabel("Frequency")
        ax.set_title(f"PRS Difference Distribution (mean={diff.mean():.4f}, std={diff.std():.4f})")
        plt.tight_layout()
        plt.savefig("benchmark_diff.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("No matching pathways between Python and R")
else:
    print("Insufficient results for comparison")
```

## Summary

```python
print("=" * 50)
print("Benchmark Summary: Python vs R")
print("=" * 50)
print(f"\nDataset: PBMC3k CD14+ Mono")
print(f"Pathways tested: {len(pathways)}")
print(f"Active pathways (>0.5): {len(pathways[pathways['pred_label'] > 0.5])}")
print(f"\nPRS Results:")
print(f"  Python: {len(prs_filtered)} pathways ({t_prs_sklearn:.2f}s)")
if len(prs_r_filtered) > 0:
    print(f"  R: {len(prs_r_filtered)} pathways ({t_prs_r:.2f}s)")
    print(f"  Speedup: {t_prs_r/t_prs_sklearn:.2f}x")
    print(f"\nCorrelation:")
    print(f"  Matching pathways: {len(merged)}")
    print(f"  Pearson: {corr_pearson:.4f}")
    print(f"  Spearman: {corr_spearman:.4f}")
    print(f"  Mean difference: {diff.mean():.6f}")
    print(f"  Std difference: {diff.std():.6f}")
print("=" * 50)
```